# Sesión 7 · Modelos de nowcasting: modelos basados en árboles

Actividad práctica · 20 minutos en vivo

Se estiman tres modelos sobre la misma base de la sesión anterior: un árbol de decisión, un
Random Forest y un Gradient Boosting. El flujo es idéntico al de la sesión 6, así que los
RMSE son directamente comparables con los de Ridge, LASSO y Elastic Net.

Las celdas marcadas con **TU TURNO** son las que se modifican en vivo.

## Preparación

La celda siguiente carga los datos, los procesa (deflactación, desestacionalización,
tasas de crecimiento), restringe a los predictores del ejercicio y arma la
partición de entrenamiento y prueba. Es idéntica en las tres actividades, así que
los resultados de las sesiones 6, 7 y 8 son directamente comparables entre sí.

Ejecutar tal cual, no hay nada que modificar.

In [ ]:
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================
# Parametros editables
# =========================================================
data_file = "data_BOL.csv"
meta_file = "meta_BOL.csv"

train_start_date = pd.to_datetime("2017-03-01")
test_start_date = pd.to_datetime("2023-09-01")

target_variable = "RGDP0000"
meta_col = "ticket"
cpi = "MCPI0000"

# Seis predictores con lectura economica clara para el PIB de Barbados.
# Para adaptar el taller a Bolivia basta con cambiar esta lista y el archivo
# de datos.
PREDICTORES = [
    "RCEM0007",   # Produccion de cemento total
    "GGTR0024",   # Google Trends: "impuestos nacionales"
    "XIMP0004",   # Importaciones de bienes de consumo duradero
    "RPES0001",   # Superficie en permisos de construccion
    "RPER0001",   # Numero de permisos de construccion
    "GGTR0010",   # Google Trends: "delivery"
    "GGTR0030",   # Google Trends: "laptop"
    "MBCB0002",   # Billetes y monedas en poder del publico
    "XFRD0014",   # Tipo de cambio BRL/USD
    "RRSU0009",   # Residuos solidos recolectados, Trinidad
    "XUSA0001",   # Exportaciones de EE.UU. a Bolivia
    "GGTR0039",   # Google Trends: "toyota"
    "RRSU0007",   # Residuos solidos recolectados, Tarija
    "XEXP0184",   # Exportaciones a Japon
    "XIMP0044",   # Importaciones de Ecuador
]

ETIQUETAS = {
    "RCEM0007": "Produccion de cemento",
    "GGTR0024": "Google Trends: impuestos nacionales",
    "XIMP0004": "Import. consumo duradero",
    "RPES0001": "Superficie permisos construccion",
    "RPER0001": "Permisos de construccion",
    "GGTR0010": "Google Trends: delivery",
    "GGTR0030": "Google Trends: laptop",
    "MBCB0002": "Billetes y monedas en poder del publico",
    "XFRD0014": "Tipo de cambio BRL/USD",
    "RRSU0009": "Residuos solidos Trinidad",
    "XUSA0001": "Export. de EE.UU. a Bolivia",
    "GGTR0039": "Google Trends: toyota",
    "RRSU0007": "Residuos solidos Tarija",
    "XEXP0184": "Export. a Japon",
    "XIMP0044": "Import. de Ecuador",
}

# Carpetas relativas, iguales a las del pipeline del taller
d = os.getcwd() + "/"
PATH_DATA = os.path.join(d, "..", "data")
PATH_OUTPUT = os.path.join(d, "..", "output")
os.makedirs(PATH_OUTPUT, exist_ok=True)

# Archivo donde se acumulan los RMSE de las tres sesiones
RESULTADOS_FILE = os.path.join(PATH_OUTPUT, "resultados_taller.csv")

# =========================================================
# Carga y procesamiento
# =========================================================
data = pd.read_csv(
    os.path.join(PATH_DATA, data_file), parse_dates=["date"], index_col="date"
)
metadata = pd.read_csv(os.path.join(PATH_DATA, meta_file))
test_end_date = data[target_variable].dropna().index.max()

# Deflactacion por IPC (base: promedio de 2017)
base = data[cpi][data.index.year == 2017].mean()
nominal = metadata.loc[metadata["nominal"] == 1][meta_col].values
for column in data.columns:
    if column in nominal:
        data[column] = data[column] / data[cpi] * base
data.drop(columns=[cpi], inplace=True)
data = data.drop(columns=[c for c in data.columns if "RGDP" in c and c != target_variable])

# Desestacionalizacion
quarterly = metadata.loc[metadata["freq"].str.lower().str.contains("q")][meta_col].values
already_sa = metadata.loc[metadata["sa"] == 1][meta_col].values
data = data.loc[:, data.notna().sum() >= 24]
for column in data.columns:
    if column in already_sa:
        continue
    serie = data[column].dropna()
    periodo = 4 if column in quarterly else 12
    decomp = seasonal_decompose(serie, model="additive", extrapolate_trend="freq", period=periodo)
    data[column] = data[column] - decomp.seasonal

# Tasas de crecimiento
for column in data.columns:
    if column in quarterly:
        no_nan = data[column].dropna()
        var = no_nan.pct_change(periods=1, fill_method=None) * 100
        res = pd.Series(np.nan, index=data.index)
        res.loc[var.index] = var
        data[column] = res
    else:
        era_nan = data[column].isna()
        var = data[column].pct_change(periods=1, fill_method=None) * 100
        prev_nan = data[column].shift(1).isna()
        var[prev_nan | era_nan] = np.nan
        data[column] = var
data.replace([np.inf, -np.inf], np.nan, inplace=True)

for column in [c for c in data.columns if c not in quarterly]:
    sub = data[data.index <= test_start_date][column]
    if sub.isna().sum() / len(sub) > 0.75:
        data.drop(column, axis=1, inplace=True)

data = data[~(data.index < train_start_date)]
data = data.dropna(how="all", axis=0)

metadata["months_lag"] = pd.to_numeric(metadata["months_lag"], errors="coerce").fillna(1).astype(int)
tickets_ok = metadata[metadata.months_lag <= 5][meta_col].tolist()
data = data[[c for c in data.columns if c in tickets_ok or "RGDP" in c]]

# =========================================================
# Restriccion a los quince predictores y agregacion trimestral
# =========================================================
columnas = [target_variable] + [c for c in PREDICTORES if c in data.columns]
faltantes = [c for c in PREDICTORES if c not in data.columns]
if faltantes:
    print(f"Aviso: no estan en la base y se omiten -> {faltantes}")
data = data[columnas]

agg_map = metadata.set_index(meta_col)["agg_method"].to_dict()
agg_funcs = {
    col: (agg_map[col] if agg_map.get(col) in ["mean", "last", "sum"] else "mean")
    for col in data.columns
}
data = data.resample("QE").agg(agg_funcs)
data = data[data.index > "2016-01-01"]
data = data[data.index < "2026-01-01"]
data = data.dropna(how="any", axis=0)

train = data[data.index < test_start_date]
test = data[data.index >= test_start_date]

X_train = train.drop(target_variable, axis=1)
y_train = train[target_variable]
X_test = test.drop(target_variable, axis=1)
y_obs = test[target_variable]

predictores = list(X_train.columns)
etiquetas = [ETIQUETAS.get(c, c) for c in predictores]

In [ ]:
def rmse(observado, predicho):
    '''Raiz del error cuadratico medio.'''
    return float(np.sqrt(np.mean((np.asarray(observado) - np.asarray(predicho)) ** 2)))


def tabla_comparativa(predicho, nombre="Predicho"):
    '''Tabla de observado contra predicho con el error en puntos porcentuales.'''
    t = pd.DataFrame({"Observado": y_obs, nombre: np.asarray(predicho)}, index=y_obs.index)
    t["Error (pp)"] = t[nombre] - t["Observado"]
    return t


def guardar_resultado(modelo, valor_rmse, sesion, notas=""):
    '''Anade o actualiza una fila en la tabla de resultados del taller.'''
    fila = pd.DataFrame([{
        "modelo": modelo,
        "rmse": round(float(valor_rmse), 4),
        "sesion": sesion,
        "notas": notas,
        "actualizado": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }])
    if os.path.exists(RESULTADOS_FILE):
        prev = pd.read_csv(RESULTADOS_FILE)
        prev = prev[prev["modelo"] != modelo]
        fila = pd.concat([prev, fila], ignore_index=True)
    fila.to_csv(RESULTADOS_FILE, index=False)
    return fila.sort_values("rmse").reset_index(drop=True)


def tabla_resultados():
    '''Devuelve todo lo acumulado hasta ahora, ordenado de mejor a peor.'''
    if not os.path.exists(RESULTADOS_FILE):
        print("Todavia no hay resultados guardados de sesiones anteriores.")
        return pd.DataFrame(columns=["modelo", "rmse", "sesion", "notas"])
    return pd.read_csv(RESULTADOS_FILE).sort_values("rmse").reset_index(drop=True)


def grafico_observado_predicho(predicho, nombre, color="#C0392B", ax=None):
    '''Serie observada contra la predicha en la muestra de prueba.'''
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2, label="Observado")
    ax.plot(y_obs.index, np.asarray(predicho), "s--", color=color, lw=2, label=nombre)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel("Crecimiento del PIB (%)")
    ax.set_title(f"Observado contra predicho · {nombre}")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    return ax


def resumen_datos():
    '''Imprime el estado de la base para verificar que todo cargo bien.'''
    print("=" * 62)
    print("BASE LISTA PARA LA ACTIVIDAD")
    print("=" * 62)
    print(f"Variable objetivo : {target_variable}")
    print(f"Predictores       : {len(predictores)} -> {', '.join(predictores)}")
    print(f"Entrenamiento     : {train.index.min():%Y-%m} a {train.index.max():%Y-%m} "
          f"({len(train)} trimestres)")
    print(f"Prueba            : {test.index.min():%Y-%m} a {test.index.max():%Y-%m} "
          f"({len(test)} trimestres)")
    print("=" * 62)
    return data.tail(5)

In [ ]:
resumen_datos()

In [ ]:
# Referencia: lo que ya se estimo en la sesion 6
tabla_resultados()

## Un solo árbol

Un árbol de regresión parte la muestra en dos una y otra vez, eligiendo cada vez la variable
y el umbral que más reducen el error dentro de los grupos resultantes. Al final, predice el
promedio del crecimiento del PIB de las observaciones que se asignaron en cada hoja.

| Parámetro | Efecto |
|---|---|
| `max_depth` | Cuántas particiones encadenadas se permiten. Más profundo implica más flexible y más propenso a memorizar el ruido. |
| `min_samples_split` | Mínimo de observaciones para poder partir un nodo. Más alto implica árboles más cortos. |
| `min_samples_leaf` | Mínimo de observaciones en una hoja. Más alto implica predicciones menos sobreajustadas. |
| `criterion` | Criterio con el que se evalúa cada partición candidata. El error absoluto es más robusto a valores atípicos que el error cuadrático. |

Con menos de treinta trimestres de entrenamiento el riesgo de sobreajuste es alto, así que los parámetros
de poda no son simplemente un detalle técnico, sino lo que permite obtener un modelo con buen desempeño.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Arbol de referencia, poco profundo, solo para tener un punto de partida
arbol_ref = DecisionTreeRegressor(
    criterion="absolute_error", max_depth=4,
    min_samples_split=6, min_samples_leaf=2, random_state=42,
).fit(X_train, y_train)

rmse_ref = rmse(y_obs, arbol_ref.predict(X_test))
print(f"RMSE con max_depth=4 (referencia): {rmse_ref:.4f}")

### Gráfico 1 · Dónde empieza el sobreajuste

En vez de ir probando profundidades una por una a mano, la celda de abajo las recorre todas
de una vez y grafica los dos errores juntos. El de entrenamiento baja siempre, porque un
árbol suficientemente profundo puede memorizar cada trimestre. El de prueba baja al
principio y luego se despega hacia arriba. El punto donde se separan es la profundidad a
partir de la cual el modelo deja de aprender y empieza a sobreajustar, y es la que conviene
usar como modelo final.

In [ ]:
import numpy as np

profundidades = [1, 2, 3, 4, 5, 6, 8, 10]
err_train, err_test = [], []

for p in profundidades:
    m = DecisionTreeRegressor(
        criterion="absolute_error", max_depth=p,
        min_samples_split=6, min_samples_leaf=2, random_state=42
    ).fit(X_train, y_train)
    err_train.append(rmse(y_train, m.predict(X_train)))
    err_test.append(rmse(y_obs, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(profundidades, err_train, "o-", color="#2980B9", lw=2, label="Entrenamiento")
ax.plot(profundidades, err_test, "s-", color="#C0392B", lw=2, label="Prueba")
ax.set_xlabel("Profundidad maxima del arbol")
ax.set_ylabel("RMSE")
ax.set_title("El arbol aprende hasta cierto punto y despues memoriza")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

pd.DataFrame({"profundidad": profundidades, "RMSE entrenamiento": err_train,
              "RMSE prueba": err_test}).round(4)

### TU TURNO: fijar el modelo final

Elegir en el gráfico de arriba la profundidad donde el error de prueba está más bajo (justo
antes de que empiece a subir) y ponerla aquí. Esta queda como el árbol oficial de la sesión,
y es también el que se dibuja y se compara más abajo.

In [ ]:
# ============ TU TURNO ============
# Copiar aqui la profundidad elegida a partir del grafico de arriba.
max_depth = 4
min_samples_leaf = 2
# ==================================

arbol = DecisionTreeRegressor(
    criterion="absolute_error",
    max_depth=max_depth,
    min_samples_split=6,
    min_samples_leaf=min_samples_leaf,
    random_state=42,
).fit(X_train, y_train)

pred_arbol = arbol.predict(X_test)
rmse_arbol = rmse(y_obs, pred_arbol)
rmse_train = rmse(y_train, arbol.predict(X_train))

print(f"RMSE en entrenamiento : {rmse_train:.4f}")
print(f"RMSE en prueba        : {rmse_arbol:.4f}")
print(f"Brecha                : {rmse_arbol - rmse_train:.4f}  <- si crece, hay sobreajuste")
print(f"Profundidad real      : {arbol.get_depth()}   (el limite era {max_depth})")
print(f"Numero de hojas       : {arbol.get_n_leaves()}")

guardar_resultado("Arbol de decision", rmse_arbol, sesion=7, notas=f"max_depth={max_depth}")

### Gráfico 2 · El árbol por dentro

Esta es la ventaja pedagógica de los árboles sobre cualquier otro modelo del taller: se
pueden dibujar y leer. Cada caja muestra la variable y el umbral con los que se partió, el
número de observaciones que llegaron ahí y el valor que el modelo predice.

Vale la pena mirar cuál variable quedó en la raíz, porque es la que más reduce el error en
una sola partición (se puede comparar mentalmente con la última que sobrevivió al LASSO en la
sesión anterior).

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(
    arbol,
    feature_names=etiquetas,
    filled=True,
    rounded=True,
    fontsize=9,
    impurity=False,
    ax=ax,
)
ax.set_title("Arbol de decision estimado sobre el PIB de Bolivia")
plt.tight_layout()

## Random Forest

En lugar de un árbol se estiman muchos, cada uno sobre una muestra bootstrap de los datos y
considerando solo un subconjunto aleatorio de variables en cada partición. Esa doble
aleatorización hace que los árboles se equivoquen de formas distintas, y al promediarlos los
errores se cancelan parcialmente.

| Parámetro | Efecto |
|---|---|
| `n_estimators` | Número de árboles. Más es mejor hasta cierto punto; por encima de 200 la ganancia es marginal. |
| `max_features` | Cuántas variables se consideran en cada partición. Es la fuente principal de diversidad entre árboles. |
| `min_samples_leaf` | Regularización individual de cada árbol. |

### TU TURNO: estimar el bosque

Se fija `max_features=3`, es decir, en cada partición cada árbol solo puede elegir entre tres
de las quince variables, sorteadas al azar. Más abajo se ve para qué sirve esa restricción.

Esta vez además se pide `oob_score=True`. El Random Forest tiene una forma de validarse a sí
mismo sin tocar la muestra de prueba: como cada árbol se entrena con una muestra bootstrap,
en promedio deja fuera cerca de un tercio de las observaciones, y esas se pueden usar para
evaluar ese árbol como si fueran datos nuevos. El promedio de esas evaluaciones es el
`oob_score_`, y sirve como una segunda opinión de qué tan bien generaliza el modelo, calculada
sin ayuda de la muestra de prueba.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# ============ TU TURNO ============
# Numero de arboles y de variables candidatas en cada particion.
n_estimators = 200
max_features = 3
# ==================================

rf = RandomForestRegressor(
    n_estimators=n_estimators,
    max_features=max_features,
    criterion="absolute_error",
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    oob_score=True,
).fit(X_train, y_train)

pred_rf = rf.predict(X_test)
rmse_rf = rmse(y_obs, pred_rf)

print(f"RMSE del Random Forest : {rmse_rf:.4f}   (arbol solo: {rmse_arbol:.4f})")
print(f"R2 fuera de bolsa (OOB): {rf.oob_score_:.3f}   <- validacion interna, sin usar la prueba")

guardar_resultado("Random Forest", rmse_rf, sesion=7,
                  notas=f"n_estimators={n_estimators}, max_features={max_features}")

### Gráfico 3 · El bosque por dentro

Cada línea gris es la predicción de uno de los 200 árboles sobre la muestra de prueba, y la
verde es su promedio, que es lo que reporta el Random Forest. Cada árbol por separado es
errático porque vio una muestra distinta y variables distintas, pero se equivocan en
direcciones diferentes y al promediarlos buena parte del error se cancela.

El panel de la derecha muestra el RMSE de cada árbol. El bosque le gana a la mayoría de sus
propios árboles, y también al árbol solo de la sección anterior, sin que ninguno de sus
componentes sea especialmente bueno.

In [ ]:
# Prediccion de cada arbol (se entrenaron sin nombres de columnas, por eso .values)
por_arbol = np.array([t.predict(X_test.values) for t in rf.estimators_])
rmse_arboles = np.array([rmse(y_obs, p) for p in por_arbol])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})

for p in por_arbol:
    ax1.plot(y_obs.index, p, color="grey", lw=0.6, alpha=0.25)
ax1.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2.5, label="Observado")
ax1.plot(y_obs.index, pred_rf, "^-", color="#27AE60", lw=2.5, label="Promedio del bosque")
ax1.axhline(0, color="grey", lw=0.8)
ax1.set_ylabel("Crecimiento del PIB (%)")
ax1.set_title("Cada linea gris es un arbol; el bosque es su promedio")
ax1.legend(frameon=False)
ax1.grid(alpha=0.25)

ax2.hist(rmse_arboles, bins=25, color="lightgrey", edgecolor="grey")
ax2.axvline(rmse_rf, color="#27AE60", lw=2.5, label=f"Bosque ({rmse_rf:.2f})")
ax2.axvline(rmse_arbol, color="#8E44AD", lw=2, ls="--", label=f"Arbol solo ({rmse_arbol:.2f})")
ax2.set_xlabel("RMSE de prueba")
ax2.set_ylabel("Numero de arboles")
ax2.set_title("Error de cada arbol del bosque")
ax2.legend(frameon=False)
plt.tight_layout()

print(f"El bosque le gana al {(rmse_arboles > rmse_rf).mean():.0%} de sus propios arboles")

## Gradient Boosting

Aquí los árboles no se promedian sino que se encadenan: cada árbol nuevo se estima sobre los
errores que dejó la suma de los anteriores, de modo que el modelo corrige sus propias fallas
etapa por etapa.

| Parámetro | Efecto |
|---|---|
| `n_estimators` | Número de etapas secuenciales. |
| `learning_rate` | Cuánto se deja aportar a cada árbol. Valores bajos exigen más etapas pero generalizan mejor. |

Estos dos parámetros se mueven juntos: bajar la tasa de aprendizaje sin subir el número de
etapas deja el modelo a medio estimar.

### Explorando la pareja de parámetros

Se prueban de una vez tres combinaciones típicas, de la más agresiva (pocas etapas, tasa
alta) a la más conservadora (muchas etapas, tasa baja).

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

combinaciones = [(0.15, 100), (0.15, 200), (0.15, 300), (0.15, 600)]

exploracion_gbt = []
for lr, n in combinaciones:
    m = GradientBoostingRegressor(
        n_estimators=n, learning_rate=lr, loss="absolute_error",
        min_samples_split=6, min_samples_leaf=3, random_state=42,
    ).fit(X_train, y_train)
    exploracion_gbt.append({
        "learning_rate": lr, "n_estimators": n,
        "RMSE": round(rmse(y_obs, m.predict(X_test)), 4),
    })

exploracion_gbt = pd.DataFrame(exploracion_gbt)
exploracion_gbt

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

combinaciones = [(0.15, 100), (0.1, 100), (0.05, 100), (0.02, 100)]

exploracion_gbt = []
for lr, n in combinaciones:
    m = GradientBoostingRegressor(
        n_estimators=n, learning_rate=lr, loss="absolute_error",
        min_samples_split=6, min_samples_leaf=3, random_state=42,
    ).fit(X_train, y_train)
    exploracion_gbt.append({
        "learning_rate": lr, "n_estimators": n,
        "RMSE": round(rmse(y_obs, m.predict(X_test)), 4),
    })

exploracion_gbt = pd.DataFrame(exploracion_gbt)
exploracion_gbt

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

combinaciones = [(0.15, 100), (0.02, 100), (0.15, 600), (0.02, 600)]

exploracion_gbt = []
for lr, n in combinaciones:
    m = GradientBoostingRegressor(
        n_estimators=n, learning_rate=lr, loss="absolute_error",
        min_samples_split=6, min_samples_leaf=3, random_state=42,
    ).fit(X_train, y_train)
    exploracion_gbt.append({
        "learning_rate": lr, "n_estimators": n,
        "RMSE": round(rmse(y_obs, m.predict(X_test)), 4),
    })

exploracion_gbt = pd.DataFrame(exploracion_gbt)
exploracion_gbt

### TU TURNO: fijar el modelo final

Copiar el par de la tabla de arriba con mejor RMSE.

In [ ]:
# ============ TU TURNO ============
# Copiar aqui el par (learning_rate, n_estimators) elegido de la tabla de arriba.
learning_rate = 0.15
n_estimators_gb = 200
# ==================================

gbt = GradientBoostingRegressor(
    n_estimators=n_estimators_gb,
    learning_rate=learning_rate,
    loss="absolute_error",
    min_samples_split=6,
    min_samples_leaf=3,
    random_state=42,
).fit(X_train, y_train)

pred_gbt = gbt.predict(X_test)
rmse_gbt = rmse(y_obs, pred_gbt)

print(f"RMSE del Gradient Boosting: {rmse_gbt:.4f}")
guardar_resultado("Gradient Boosting", rmse_gbt, sesion=7,
                  notas=f"lr={learning_rate}, n_estimators={n_estimators_gb}")

### Gráfico 4 · Entrenamiento contra validación, ronda por ronda

El error en entrenamiento de un Gradient Boosting siempre baja, porque agregar árboles nunca
empeora el ajuste sobre los datos que el modelo ya vio. La pregunta que importa es otra, en
qué ronda el modelo deja de mejorar sobre datos que no ha visto.

Para responderla se usa `staged_predict()`, que en vez de devolver solo la predicción final
con todos los árboles ya sumados va devolviendo la predicción acumulada después de cada
ronda, como si se pausara el entrenamiento en cada punto. Con eso se calcula el RMSE en
ambas muestras para cada ronda y se grafican las dos curvas juntas.

El resultado es la figura canónica del método, la misma de la teoría: la curva de
entrenamiento cae y se aplana, la de prueba baja, toca un mínimo y empieza a subir. Ese
mínimo es la ronda de parada temprana (`early stopping`), y todo lo que viene después es el
modelo memorizando las particularidades de los trimestres de entrenamiento.

Se corre a propósito con muchas más rondas de las necesarias, para que el tramo de
sobreajuste se vea completo.

In [ ]:
n_rondas_largo = 500

gbt_largo = GradientBoostingRegressor(
    n_estimators=n_rondas_largo,
    learning_rate=0.15,
    loss="absolute_error",
    min_samples_split=6,
    min_samples_leaf=3,
    random_state=42,
).fit(X_train, y_train)

# staged_predict devuelve la prediccion acumulada despues de cada ronda
err_train_r = [rmse(y_train, p) for p in gbt_largo.staged_predict(X_train)]
err_test_r = [rmse(y_obs, p) for p in gbt_largo.staged_predict(X_test)]

ronda_optima = int(np.argmin(err_test_r)) + 1
print(f"Ronda con menor RMSE de prueba: {ronda_optima} de {n_rondas_largo}")
print(f"RMSE en esa ronda            : {min(err_test_r):.4f}")
print(f"RMSE en la ronda final       : {err_test_r[-1]:.4f}   <- peor, por sobreajuste")

In [ ]:
rondas = range(1, n_rondas_largo + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rondas, err_train_r, color="#2980B9", lw=1.8, label="Entrenamiento")
ax.plot(rondas, err_test_r, color="#E67E22", lw=1.8, label="Prueba")
ax.axvline(ronda_optima, color="#27AE60", ls="--", lw=2)
ax.plot(ronda_optima, min(err_test_r), "o", color="#27AE60", markersize=10)
ax.annotate(f"parada temprana\n(ronda {ronda_optima})",
            xy=(ronda_optima, min(err_test_r)),
            xytext=(ronda_optima + n_rondas_largo * 0.12, min(err_test_r) * 1.5),
            color="#27AE60", fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#27AE60"))

ax.axvspan(0, ronda_optima, color="#27AE60", alpha=0.06)
ax.axvspan(ronda_optima, n_rondas_largo, color="#E67E22", alpha=0.06)

ax.set_xlabel("Ronda de boosting")
ax.set_ylabel("RMSE")
ax.set_title("Gradient Boosting: entrenamiento contra prueba")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

En producción no hace falta leer este gráfico a ojo para después recortar el modelo a mano.
`GradientBoostingRegressor` trae `n_iter_no_change` y `validation_fraction`, que apartan una
fracción del entrenamiento como validación interna y detienen el boosting solos cuando la
pérdida deja de mejorar durante un número dado de rondas. Aquí se hizo a mano porque el
punto de la actividad es ver la forma de las dos curvas, no automatizar el corte.

Una advertencia sobre este gráfico en particular: la curva naranja está calculada sobre la
misma muestra de prueba que se usa para reportar el RMSE final, así que elegir la ronda
mirándola es hacer trampa en sentido estricto. Sirve para entender el fenómeno, no para
calibrar un modelo que se vaya a reportar.

### Gráfico 5 · Las tres predicciones sobre la muestra de prueba

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2.5, label="Observado")
ax.plot(y_obs.index, pred_arbol, "s--", color="#8E44AD", lw=1.8, label="Arbol")
ax.plot(y_obs.index, pred_rf, "^--", color="#27AE60", lw=1.8, label="Random Forest")
ax.plot(y_obs.index, pred_gbt, "d--", color="#E67E22", lw=1.8, label="Gradient Boosting")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("Crecimiento del PIB (%)")
ax.set_title("Modelos de arboles en la muestra de prueba")
ax.legend(frameon=False, ncol=4)
ax.grid(alpha=0.25)
plt.tight_layout()

## Cierre: tabla acumulada

In [ ]:
tabla_resultados()

### Gráfico Extra · Importancia de variables frente a los coeficientes del LASSO

Este gráfico cierra el arco de las dos sesiones. Los modelos de árboles reportan cuánto
redujo el error cada variable a lo largo de todas las particiones en las que participó, que
es una forma de medir su aporte sin imponer ninguna forma funcional. El LASSO llega a la
misma pregunta por un camino opuesto, eliminando coeficientes de un modelo lineal.

Cuando ambos coinciden en las mismas dos o tres series, el resultado es sólido y se puede
defender ante una autoridad monetaria. Cuando difieren hay dos explicaciones posibles y vale
la pena distinguirlas: que la variable entre al PIB de forma no lineal, y entonces el modelo
lineal no la estaba viendo, o que la penalización de LASSO la haya castigado por ser poco
volátil, cosa que a los árboles no les pasa porque parten por umbrales. La tabla de
desviaciones típicas de la sesión anterior ayuda a decidir cuál de las dos es.

#### Qué es la importancia de variables y cómo se calcula

Un árbol funciona ordenando trimestres. Arranca con todos juntos, donde el crecimiento del PIB
es muy disperso, y en cada paso los separa en dos grupos usando una variable y un umbral, por
ejemplo trimestres con exportaciones creciendo por encima de 2% y trimestres por debajo. Si la
separación es buena, dentro de cada grupo el PIB se parece más entre sí que antes de partir.
La importancia de una variable responde a una pregunta simple: de todo lo que el modelo logró
ordenar, qué parte se debe a esa variable.

**El aporte de una partición.** Para medir qué tan desordenado está un grupo se usa la
*impureza*, que en este notebook (`criterion="absolute_error"`) es la distancia promedio del
crecimiento del PIB a la mediana del grupo, en puntos porcentuales. Si se usara
`criterion="squared_error"` sería la varianza. Cuando un nodo $t$ con $N_t$ trimestres se
parte en un grupo izquierdo $L$ y uno derecho $R$, la reducción de desorden es

$$
\Delta I(t) = \frac{N_t}{N}\left[\, I(t) - \frac{N_L}{N_t}\, I(L) - \frac{N_R}{N_t}\, I(R) \,\right]
$$

El corchete compara la impureza antes de partir con la impureza promedio de los dos grupos
después de partir, cada uno pesado por su tamaño. Es exactamente el mismo criterio que usa el
árbol para decidir cómo partir: entre todas las variables y umbrales posibles elige el que hace
más grande ese corchete. La importancia, entonces, no calcula nada nuevo, solo lleva la cuenta
de lo que ganó cada decisión que el árbol ya tomó. El factor $N_t/N$, donde $N$ es el total de
trimestres, hace que una partición que afecta a toda la muestra pese más que una que afecta a
tres o cuatro trimestres en el fondo del árbol.

**De particiones a importancia.** Se suman los aportes de todas las particiones hechas con cada
variable, se promedia entre los árboles del bosque y se divide por el total para que todas las
importancias sumen uno:

$$
\text{Imp}_j = \frac{1}{B}\sum_{b=1}^{B}\;\sum_{t \,\in\, b \,:\, v(t)=j} \Delta I(t),
\qquad \sum_j \text{Imp}_j = 1
$$

donde $B$ es el número de árboles y $v(t)$ la variable usada en el nodo $t$.

**Un ejemplo con números redondos.** Al inicio, el crecimiento del PIB de los trimestres de
entrenamiento se aleja en promedio 1.5 pp de su mediana. El árbol parte primero por
exportaciones totales y, dentro de los dos grupos que resultan, esa distancia promedio baja a
1.0 pp. La partición aporta 0.5 pp, con peso completo porque involucra a todos los trimestres.
Después, en un grupo que contiene un tercio de la muestra, parte por base monetaria y la
distancia baja de 0.9 a 0.6 pp, lo que aporta 0.3 × 1/3 = 0.1 pp. El árbol redujo en total
0.6 pp, de los cuales 0.5 vinieron de exportaciones y 0.1 de base monetaria, así que sus
importancias son 0.83 y 0.17.

**Cómo leer el número.** Una importancia de 0.83 significa que el 83% de toda la reducción de
dispersión del PIB que logró el modelo vino de particiones sobre esa variable. Habla de cuánto
la usó el modelo para separar trimestres de crecimiento alto y bajo, y no dice nada de cuánto
se mueve el PIB cuando la variable cambia. Tampoco tiene signo: una variable puede ser muy
importante sin que el gráfico diga si se asocia con más o con menos crecimiento.

En el Gradient Boosting el cálculo es el mismo, salvo que cada árbol ordena los errores que
dejaron los anteriores, así que la importancia mide cuánto ayudó cada variable a corregirlos.
Para poner al LASSO en el mismo gráfico se normalizan sus coeficientes en valor absoluto,
$|\beta_j| / \sum_k |\beta_k|$, pero hay que tener presente que miden cosas distintas: el
coeficiente es un efecto (cuántos puntos de PIB por punto de la variable) y la importancia es
una contribución a ordenar los datos.

**Cautelas.** Como las importancias siempre suman uno, un modelo que apenas reduce el error
igual reparte el 100% entre sus variables, así que hay que leerlas junto con el RMSE. Se
calculan sobre la muestra de entrenamiento, de modo que una variable que el árbol usó para
memorizar ruido también acumula importancia. Cuando dos series están muy correlacionadas, como
la llegada y la salida de visitantes, los árboles las usan casi indistintamente y cada una
aparece menos importante de lo que es en conjunto. Y con menos de treinta trimestres el orden
puede cambiar si se mueve la semilla aleatoria, por lo que lo sensato es fijarse en qué
variables quedan arriba y no en diferencias pequeñas entre ellas.

In [ ]:
from sklearn.linear_model import Lasso

# Se reestima el LASSO de la sesion 6 para poder comparar lado a lado
lasso = Lasso(alpha=0.05, max_iter=10000).fit(X_train, y_train)

importancias = pd.DataFrame({
    "Random Forest": rf.feature_importances_,
    "Gradient Boosting": gbt.feature_importances_,
}, index=etiquetas)

# Los coeficientes del LASSO se llevan a magnitud relativa para que las escalas
# sean visualmente comparables
coef_abs = np.abs(lasso.coef_)
importancias["LASSO (coef. relativo)"] = (
    coef_abs / coef_abs.sum() if coef_abs.sum() > 0 else coef_abs
)

orden = importancias["Random Forest"].sort_values().index
imp = importancias.loc[orden]

fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(imp))
alto = 0.26
paleta = ["#27AE60", "#E67E22", "#C0392B"]

for i, (col, color) in enumerate(zip(imp.columns, paleta)):
    ax.barh(y + i * alto - alto, imp[col], alto, label=col, color=color)

ax.set_yticks(y)
ax.set_yticklabels(imp.index)
ax.set_xlabel("Importancia relativa")
ax.set_title("Tres modelos, una misma pregunta: que series informan sobre el PIB")
ax.legend(frameon=False)
ax.grid(alpha=0.25, axis="x")
plt.tight_layout()

importancias.round(3)